# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

       impressions_90d  days_since_last_update  ...   ctr  trend_direction
16751            61678                     194  ...  0.15             down
16514            59472                     194  ...  0.13             down
7021             25715                     194  ...  0.23             down
21268            13299                     193  ...  0.49             down
11489             7812                     194  ...  0.01             down
12045             7558                     193  ...  0.20             down
698               4590                     194  ...  0.00             down
5327              4556                     194  ...  0.33             down
26810             4429                     194  ...  0.38             down
20837             1697                     193  ...  0.12             down

[10 rows x 5 columns]

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.950
Hand rule  Precision@50: 0.660


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.950   vs   tree 0.650
Precision@50:  hand rule 0.660   vs   tree 0.640


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Your experiment here
from sklearn.model_selection import GroupShuffleSplit

# Client-holdout split: about 20% of clients go to test, none of their pages are seen in training
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(split.split(df, y, groups=df["client_id"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]
print(f"train: {len(train)} pages / {train['client_id'].nunique()} clients | "
      f"test: {len(test)} pages / {test['client_id'].nunique()} clients | "
      f"test declining rate {test['is_declining_label'].mean():.3f}")

def fit_and_score(feats, depth):
    clean = lambda d: d[feats].replace([np.inf, -np.inf], np.nan).fillna(0)
    m = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    m.fit(clean(train), train["is_declining_label"])
    s = m.predict_proba(clean(test))[:, 1]
    return m, [precision_at_k(s, test["is_declining_label"], k) for k in (20, 50, 200)]

print("\nHand rule on test  P@20 / P@50 / P@200:",
      " / ".join(f"{precision_at_k(test['hand_rule_score'], test['is_declining_label'], k):.3f}" for k in (20, 50, 200)))

swapped = [f for f in features if f != "impressions_90d"] + ["engagement_rate"]
for name, feats in [("original features", features), ("no impressions, + engagement_rate", swapped)]:
    for depth in (2, 3, 4):
        m, p = fit_and_score(feats, depth)
        first = feats[m.tree_.feature[0]]
        print(f"{name:34s} depth {depth}: P@20 {p[0]:.3f}  P@50 {p[1]:.3f}  P@200 {p[2]:.3f}  "
              f"| leaves {m.get_n_leaves():2d} | first split: {first}")

m3, _ = fit_and_score(features, 3)
print("\nDepth-3 tree, trained on train clients only:")
print(export_text(m3, feature_names=features))


train: 23837 pages / 25 clients | test: 6163 pages / 7 clients | test declining rate 0.511

Hand rule on test  P@20 / P@50 / P@200: 0.800 / 0.700 / 0.575
original features                  depth 2: P@20 0.700  P@50 0.720  P@200 0.630  | leaves  4 | first split: impressions_90d
original features                  depth 3: P@20 0.700  P@50 0.660  P@200 0.600  | leaves  8 | first split: impressions_90d
original features                  depth 4: P@20 0.550  P@50 0.620  P@200 0.620  | leaves 16 | first split: impressions_90d
no impressions, + engagement_rate  depth 2: P@20 0.550  P@50 0.600  P@200 0.595  | leaves  4 | first split: avg_position
no impressions, + engagement_rate  depth 3: P@20 0.500  P@50 0.560  P@200 0.590  | leaves  8 | first split: avg_position
no impressions, + engagement_rate  depth 4: P@20 0.700  P@50 0.600  P@200 0.575  | leaves 16 | first split: avg_position

Depth-3 tree, trained on train clients only:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   | 

### What I found

On a client-holdout split (25 train clients, 7 test clients, 6,163 test pages, 51% declining), the hand rule still holds the very top of the list: P@20 0.80, P@50 0.70. The depth-2 tree scores 0.70 and 0.72. Going deeper did not help. Depth 3 gave P@50 0.66 and depth 4 gave 0.62. I can still read the depth-3 tree; the depth-4 one has 16 leaves and is hard to follow.

When I dropped `impressions_90d` and added `engagement_rate`, the tree split on `avg_position` first and P@50 fell to 0.56-0.60. Impressions carry most of what these small trees learn.

Caveats. The test set is only 7 clients, so another seed could move these numbers a lot. A depth-2 tree has only 4 distinct scores, so its top 50 is a block of tied pages and the order inside it is arbitrary. The hand rule is not fitted, so its drop from 0.95 to 0.80 at P@20 comes from scoring a different set of pages; the trees, on the other hand, were fitted on train clients only, so these are their honest held-out numbers. All of this is observed on this 30,000-page sample only.


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.